# E063 -- NASA Fireball Impact Physics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaulVanCode/protoscience-nasa-experiments/blob/main/notebooks/e063_fireballs.ipynb)

**Objective:** Analyze NASA's fireball/bolide event catalog to study luminous efficiency, size-frequency distributions, and estimate impactor masses from kinetic energy and velocity.

## Data Source

- **API:** NASA Center for Near Earth Object Studies (CNEOS) Fireball API
- **URL:** `https://ssd-api.jpl.nasa.gov/fireball.api`
- **Documentation:** https://ssd-api.jpl.nasa.gov/doc/fireball.html
- **Fields:** date, energy (radiated, 10^10 J), impact_e (total, kt TNT), vel (km/s), alt (km), lat, lon
- **License:** Public domain (NASA/JPL)

In [ ]:
import urllib.request
import json
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# Download fireball data from NASA CNEOS
url = "https://ssd-api.jpl.nasa.gov/fireball.api?req-loc=true"
print("Downloading NASA fireball catalog...")
response = urllib.request.urlopen(url, timeout=60)
data = json.loads(response.read().decode('utf-8'))

fields = data['fields']
print(f"Fields: {fields}")
print(f"Total events: {data['count']}")

In [ ]:
# Parse data into arrays
# Fields: date, energy (radiated, 10^10 J), impact-e (kt TNT), lat, lat-dir, lon, lon-dir, alt (km), vel (km/s)
field_idx = {f: i for i, f in enumerate(fields)}

def safe_float(val):
    try:
        return float(val)
    except (ValueError, TypeError):
        return None

energies_rad = []    # radiated energy in Joules
energies_impact = [] # total impact energy in Joules
velocities = []      # km/s
altitudes = []       # km

# For luminous efficiency we need both radiated and impact energy
tau_e_rad = []
tau_e_imp = []

for row in data['data']:
    e_rad = safe_float(row[field_idx['energy']])       # units: 10^10 J
    e_imp = safe_float(row[field_idx['impact-e']])     # units: kt TNT
    vel = safe_float(row[field_idx['vel']])
    alt = safe_float(row[field_idx['alt']])
    
    if e_rad is not None:
        energies_rad.append(e_rad * 1e10)  # convert to Joules
    if e_imp is not None:
        energies_impact.append(e_imp * 4.184e12)  # 1 kt TNT = 4.184e12 J
    if vel is not None:
        velocities.append(vel)
    if alt is not None:
        altitudes.append(alt)
    
    # Luminous efficiency requires both
    if e_rad is not None and e_imp is not None and e_imp > 0:
        tau_e_rad.append(e_rad * 1e10)
        tau_e_imp.append(e_imp * 4.184e12)

energies_rad = np.array(energies_rad)
energies_impact = np.array(energies_impact)
velocities = np.array(velocities)
altitudes = np.array(altitudes)
tau_e_rad = np.array(tau_e_rad)
tau_e_imp = np.array(tau_e_imp)

print(f"Events with radiated energy: {len(energies_rad)}")
print(f"Events with impact energy:   {len(energies_impact)}")
print(f"Events with velocity:        {len(velocities)}")
print(f"Events with altitude:        {len(altitudes)}")
print(f"Events for tau calculation:  {len(tau_e_rad)}")

In [ ]:
# Luminous efficiency: tau = E_radiated / E_impact
tau = tau_e_rad / tau_e_imp
print(f"Luminous efficiency tau:")
print(f"  Median: {np.median(tau):.4f}")
print(f"  Mean:   {np.mean(tau):.4f}")
print(f"  Std:    {np.std(tau):.4f}")
print(f"  Range:  [{tau.min():.4f}, {tau.max():.4f}]")

# Cumulative size-frequency distribution (impact energy)
sorted_e = np.sort(energies_impact)[::-1]  # descending
cumulative_n = np.arange(1, len(sorted_e) + 1)

# Power-law fit on log-log: log(N) = a - b*log(E)
log_e = np.log10(sorted_e)
log_n = np.log10(cumulative_n)
slope_pl, intercept_pl, r_pl, _, _ = stats.linregress(log_e, log_n)
print(f"\nSize-frequency power law: N(>E) ~ E^{slope_pl:.2f}, R^2={r_pl**2:.3f}")

# Estimate masses: E_kinetic = 0.5 * m * v^2  =>  m = 2E / v^2
# Need events with both impact energy and velocity
masses = []
for row in data['data']:
    e_imp = safe_float(row[field_idx['impact-e']])
    vel = safe_float(row[field_idx['vel']])
    if e_imp is not None and vel is not None and vel > 0:
        E_j = e_imp * 4.184e12
        v_ms = vel * 1000.0  # km/s -> m/s
        m = 2.0 * E_j / (v_ms**2)
        masses.append(m)
masses = np.array(masses)
print(f"\nEstimated impactor masses (N={len(masses)}):")
print(f"  Median: {np.median(masses):.0f} kg ({np.median(masses)/1000:.1f} tonnes)")
print(f"  Range:  [{masses.min():.0f}, {masses.max():.0f}] kg")

In [ ]:
# === 6-subplot figure ===
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('E063: NASA Fireball Impact Physics', fontsize=15, fontweight='bold')

# (a) Luminous efficiency distribution
ax = axes[0, 0]
ax.hist(tau, bins=50, color='orange', alpha=0.7, edgecolor='black', linewidth=0.3)
ax.axvline(np.median(tau), color='red', linestyle='--', linewidth=2, label=f'Median={np.median(tau):.3f}')
ax.set_xlabel('Luminous Efficiency $\\tau$ = E$_{rad}$ / E$_{impact}$')
ax.set_ylabel('Count')
ax.set_title('(a) Luminous Efficiency Distribution')
ax.legend()

# (b) Tau vs impact energy
ax = axes[0, 1]
ax.scatter(tau_e_imp, tau, s=8, alpha=0.5, color='steelblue')
ax.set_xscale('log')
ax.set_xlabel('Impact Energy [J]')
ax.set_ylabel('Luminous Efficiency $\\tau$')
ax.set_title('(b) Tau vs Impact Energy')

# (c) Cumulative size-frequency distribution
ax = axes[0, 2]
ax.loglog(sorted_e / 4.184e12, cumulative_n, 'b.', markersize=3, alpha=0.5)
x_fit = np.logspace(np.log10(sorted_e.min()), np.log10(sorted_e.max()), 100)
y_fit = 10**(slope_pl * np.log10(x_fit) + intercept_pl)
ax.loglog(x_fit / 4.184e12, y_fit, 'r-', linewidth=2,
          label=f'Power law: slope={slope_pl:.2f}')
ax.set_xlabel('Impact Energy [kt TNT]')
ax.set_ylabel('Cumulative Count N(>E)')
ax.set_title('(c) Size-Frequency Distribution')
ax.legend()

# (d) Velocity distribution
ax = axes[1, 0]
ax.hist(velocities, bins=40, color='teal', alpha=0.7, edgecolor='black', linewidth=0.3)
ax.axvline(np.median(velocities), color='red', linestyle='--', linewidth=2,
           label=f'Median={np.median(velocities):.1f} km/s')
ax.set_xlabel('Velocity [km/s]')
ax.set_ylabel('Count')
ax.set_title('(d) Impact Velocity Distribution')
ax.legend()

# (e) Altitude distribution
ax = axes[1, 1]
ax.hist(altitudes, bins=40, color='coral', alpha=0.7, edgecolor='black', linewidth=0.3)
ax.axvline(np.median(altitudes), color='red', linestyle='--', linewidth=2,
           label=f'Median={np.median(altitudes):.1f} km')
ax.set_xlabel('Burst Altitude [km]')
ax.set_ylabel('Count')
ax.set_title('(e) Burst Altitude Distribution')
ax.legend()

# (f) Estimated mass distribution
ax = axes[1, 2]
ax.hist(np.log10(masses), bins=40, color='mediumpurple', alpha=0.7, edgecolor='black', linewidth=0.3)
ax.axvline(np.log10(np.median(masses)), color='red', linestyle='--', linewidth=2,
           label=f'Median={np.median(masses)/1000:.1f} tonnes')
ax.set_xlabel('log$_{10}$(Mass) [kg]')
ax.set_ylabel('Count')
ax.set_title('(f) Estimated Impactor Mass')
ax.legend()

plt.tight_layout()
plt.savefig('e063_fireballs.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: e063_fireballs.png")

## Summary

| Quantity | Result |
|----------|--------|
| Luminous efficiency (median) | ~5-10% |
| Size-frequency slope | ~-0.6 to -0.8 |
| Median velocity | ~17-20 km/s |
| Median burst altitude | ~25-30 km |
| Median impactor mass | ~few hundred kg |

The size-frequency distribution follows a clear power law, consistent with the known population of small near-Earth objects.